# 📊 Notebook 05 — Power BI Dashboard
### Sales & Retail Analytics Portfolio Project
**Author:** Muhammad Umar Shahid
**Email:** connect2umarshahid@gmail.com
**GitHub:** [github.com/Umar-Shahid/sales-retail-analytics](https://github.com/Umar-Shahid/sales-retail-analytics)

---

## 📋 What This Notebook Covers

This notebook is a **step-by-step companion guide** for building the
Power BI dashboard. It documents every decision so the dashboard is
fully reproducible.

| Section | Content |
|---------|---------|
| 1 | Files to load & data model setup |
| 2 | Relationships between tables |
| 3 | DAX measures (copy-paste ready) |
| 4 | Dashboard pages & visuals to build |
| 5 | Formatting & publishing guide |

## 🗂️ Dashboard Pages We Will Build

| Page | Title | Purpose |
|------|-------|---------|
| 1 | 📈 Executive Summary | KPIs, sales trend, profit by year |
| 2 | 🌍 Regional Analysis | Sales & profit by region and country |
| 3 | 📦 Product Performance | Category, sub-category, top products |
| 4 | 👥 Customer Segments | RFM segments, top customers |
| 5 | ⚠️ Churn Risk | Risk tiers, at-risk revenue by region |
| 6 | 🔮 Sales Forecast | Prophet forecast imported as visual |

## ⚙️ Requirements
- Power BI Desktop (free) — download from microsoft.com/powerbi
- All files from `data/processed/` and `database/` folders

---
## 📥 Section 1 — Loading Data into Power BI

### Step-by-Step: Import All Files

#### Step 1 — Open Power BI Desktop
- Launch Power BI Desktop
- Click **"Get Data"** → **"Text/CSV"**

#### Step 2 — Load These Files in Order

| # | File | Location | Table Name in Power BI |
|---|------|----------|----------------------|
| 1 | `superstore_clean.csv` | `data/processed/` | `Orders` |
| 2 | `rfm_segments.csv` | `data/processed/` | `RFM_Segments` |
| 3 | `churn_risk_scores.csv` | `data/processed/` | `Churn_Scores` |

#### Step 3 — Load SQLite Database (Optional)
Power BI can connect directly to SQLite via ODBC:
- Get Data → ODBC
- DSN → point to `database/superstore.db`
- Import tables: `customers`, `products`, `orders`,
  `rfm_segments`, `churn_scores`

> **Recommended:** Use CSV files for simplicity.
> SQLite ODBC requires driver installation.

#### Step 4 — Fix Date Columns
After loading `superstore_clean.csv` in Power Query:
- Select `Order Date` column
- Home → Data Type → **Date**
- Select `Ship Date` column
- Home → Data Type → **Date**
- Click **Close & Apply**

#### Step 5 — Verify Load
After importing check:

| Table | Expected Rows | Key Columns |
|-------|--------------|-------------|
| Orders | 8,286 | Order Date, Sales, Profit, Discount |
| RFM_Segments | ~1,590 | Customer ID, Segment, Monetary |
| Churn_Scores | ~1,590 | Customer ID, Churn_Proba, Risk_Tier |

### ⚠️ Common Issues

| Problem | Fix |
|---------|-----|
| Dates showing as text | Change data type in Power Query |
| Decimal separator wrong | File → Options → Regional Settings → change to English (US) |
| Blank rows at bottom | Power Query → Remove Bottom Rows → 1 |

---
## 🔗 Section 2 — Data Model & Relationships

### What is a Data Model in Power BI?
A data model defines how tables are connected to each other.
When tables are related, a filter on one table automatically
filters all connected tables — this powers slicers and cross-filtering.

---

### Our Star Schema

```
                    ┌──────────────────────┐
                    │     RFM_Segments     │
                    │──────────────────────│
                    │ customer_id      PK  │
                    │ customer_name        │
                    │ segment              │
                    │ recency              │
                    │ frequency            │
                    │ monetary             │
                    └──────────┬───────────┘
                               │ 1
                               │
┌──────────────────────┐       │       ┌──────────────────────┐
│     Churn_Scores     │       │       │        Orders        │
│──────────────────────│     * │ *     │──────────────────────│
│ customer_id      PK  ├───────┴───────┤ customer_id      FK  │
│ customer_name        │               │ order_id             │
│ recency              │               │ order_date           │
│ frequency            │               │ ship_date            │
│ monetary             │               │ ship_mode            │
│ churn_proba          │               │ sales                │
│ risk_tier            │               │ quantity             │
└──────────────────────┘               │ discount             │
                                       │ profit               │
                                       │ category             │
                                       │ region               │
                                       │ season               │
                                       └──────────────────────┘
```

---

### Relationship Summary

| From Table   | Column      | To Table | Column      | Cardinality  |
|--------------|-------------|----------|-------------|--------------|
| RFM_Segments | customer_id | Orders   | customer_id | One to Many  |
| Churn_Scores | customer_id | Orders   | customer_id | One to Many  |

---

### How to Set Up Relationships (Step by Step)

**Step 1** — Click **Model** view in the left sidebar (third icon)

**Step 2** — Three table cards will appear on the canvas

**Step 3** — Drag `customer_id` from `RFM_Segments` → drop onto `customer_id` in `Orders`

**Step 4** — Drag `customer_id` from `Churn_Scores` → drop onto `customer_id` in `Orders`

**Step 5** — Double-click each relationship line and verify:

| Setting | Value |
|---------|-------|
| Cardinality | One to Many (1 : *) |
| Cross filter direction | Single |
| Active | ✅ Checked |

---

### Why Single Cross-Filter Direction?

| Direction | Behaviour |
|-----------|-----------|
| RFM_Segments → Orders | ✅ Correct — segment filter drills into orders |
| Churn_Scores → Orders | ✅ Correct — risk tier filter drills into orders |
| Orders → Dimensions | ❌ Disabled — prevents circular dependencies |

> Bidirectional filtering causes ambiguous results and slow performance.
> Always use **Single** direction in a star schema.

---
## 📐 Section 3 — DAX Measures

### What is DAX?
**DAX** (Data Analysis Expressions) is Power BI's formula language.
DAX measures are calculated dynamically based on whatever filters
or slicers the user applies — they are not stored columns.

```
Stored Column  → calculated once when data loads, fixed value
DAX Measure    → recalculated every time a filter changes, dynamic
```

### How to Add a Measure
1. Click on the **Orders** table in the Fields pane
2. Home → **New Measure**
3. Paste the DAX formula
4. Press **Enter**
5. Rename the measure in the formula bar

> Create all measures inside the **Orders** table
> unless specified otherwise.

---

### 📊 Page 1 — Executive Summary Measures

```dax
-- Total Sales
Total Sales = SUM(Orders[sales])

-- Total Profit
Total Profit = SUM(Orders[profit])

-- Total Orders
Total Orders = DISTINCTCOUNT(Orders[order_id])

-- Total Customers
Total Customers = DISTINCTCOUNT(Orders[customer_id])

-- Overall Profit Margin %
Profit Margin % =
DIVIDE(
    SUM(Orders[profit]),
    SUM(Orders[sales]),
    0
) * 100

-- Average Order Value
Avg Order Value =
DIVIDE(
    SUM(Orders[sales]),
    DISTINCTCOUNT(Orders[order_id]),
    0
)

-- Year over Year Sales Growth %
YoY Sales Growth % =
VAR CurrentYear = SUM(Orders[sales])
VAR PreviousYear =
    CALCULATE(
        SUM(Orders[sales]),
        SAMEPERIODLASTYEAR(Orders[order_date])
    )
RETURN
DIVIDE(
    CurrentYear - PreviousYear,
    PreviousYear,
    0
) * 100
```

---

### 🌍 Page 2 — Regional Analysis Measures

```dax
-- Sales by Region (used in map visual)
Regional Sales = SUM(Orders[sales])

-- Profit by Region
Regional Profit = SUM(Orders[profit])

-- Regional Profit Margin %
Regional Margin % =
DIVIDE(
    SUM(Orders[profit]),
    SUM(Orders[sales]),
    0
) * 100

-- Top Region (for KPI card)
Top Region =
CALCULATE(
    FIRSTNONBLANK(Orders[Region], 1),
    TOPN(
        1,
        SUMMARIZE(Orders, Orders[Region], "RegSales", SUM(Orders[sales])),
        [RegSales], DESC
    )
)
```

---

### 📦 Page 3 — Product Performance Measures

```dax
-- Total Quantity Sold
Total Quantity = SUM(Orders[quantity])

-- Avg Discount %
Avg Discount % =
AVERAGE(Orders[discount]) * 100

-- Avg Profit per Order
Avg Profit per Order =
DIVIDE(
    SUM(Orders[profit]),
    DISTINCTCOUNT(Orders[order_id]),
    0
)

-- Loss-Making Orders Count
Loss Orders =
CALCULATE(
    DISTINCTCOUNT(Orders[order_id]),
    Orders[profit] < 0
)

-- Loss Orders %
Loss Orders % =
DIVIDE(
    [Loss Orders],
    [Total Orders],
    0
) * 100
```

---

### 👥 Page 4 — Customer Segment Measures

```dax
-- Champions Revenue
Champions Revenue =
CALCULATE(
    SUM(Orders[sales]),
    RFM_Segments[segment] = "🏆 Champions"
)

-- Loyal Customers Revenue
Loyal Revenue =
CALCULATE(
    SUM(Orders[sales]),
    RFM_Segments[segment] = "💛 Loyal Customers"
)

-- At Risk Revenue
At Risk Revenue =
CALCULATE(
    SUM(Orders[sales]),
    RFM_Segments[segment] = "⚠️  At Risk"
)

-- Lost Revenue
Lost Revenue =
CALCULATE(
    SUM(Orders[sales]),
    RFM_Segments[segment] = "❌ Lost / Inactive"
)

-- Avg Monetary Value per Segment
Avg Customer Value =
DIVIDE(
    SUM(Orders[sales]),
    DISTINCTCOUNT(Orders[customer_id]),
    0
)
```

---

### ⚠️ Page 5 — Churn Risk Measures

```dax
-- Critical Risk Customer Count
Critical Risk Count =
CALCULATE(
    DISTINCTCOUNT(Churn_Scores[customer_id]),
    Churn_Scores[risk_tier] = "🔴 Critical Risk"
)

-- High Risk Customer Count
High Risk Count =
CALCULATE(
    DISTINCTCOUNT(Churn_Scores[customer_id]),
    Churn_Scores[risk_tier] = "🟠 High Risk"
)

-- Revenue at Risk (Critical + High)
Revenue at Risk =
CALCULATE(
    SUM(Orders[sales]),
    Churn_Scores[risk_tier] IN {
        "🔴 Critical Risk",
        "🟠 High Risk"
    }
)

-- Avg Churn Probability %
Avg Churn Probability =
AVERAGE(Churn_Scores[churn_proba]) * 100

-- Safe Customer Count
Safe Customers =
CALCULATE(
    DISTINCTCOUNT(Churn_Scores[customer_id]),
    Churn_Scores[risk_tier] IN {
        "🟢 Low Risk",
        "🟡 Medium Risk"
    }
)
```

---

### 🎨 DAX Best Practices

| Rule | Reason |
|------|--------|
| Use `DIVIDE(a, b, 0)` not `a / b` | Prevents division by zero errors |
| Use `DISTINCTCOUNT` for orders/customers | Avoids double counting |
| Use `CALCULATE` to override filters | Most powerful DAX function |
| Name measures clearly | `Total Sales` not `ts` |
| Group measures in a dedicated table | Easier to find in Fields pane |

---
## 🎨 Section 4 — Dashboard Pages & Visuals Guide

### General Formatting Rules (Apply to All Pages)
- Canvas size       : **1280 × 720** (16:9 widescreen)
- Background color  : **#F8F9FA** (light gray)
- Font family       : **Segoe UI** (Power BI default)
- Title font size   : **16pt Bold**
- Accent color      : **#2C3E50** (dark navy)
- Highlight color   : **#E74C3C** (red)
- Positive color    : **#2ECC71** (green)
- Negative color    : **#E74C3C** (red)

---

### 📄 Page 1 — Executive Summary

**Purpose:** Give leadership a full business snapshot at a glance.

#### Visuals to Add

| Visual Type | Fields | Position |
|-------------|--------|----------|
| Card | Total Sales | Top row — left |
| Card | Total Profit | Top row — center left |
| Card | Profit Margin % | Top row — center right |
| Card | Total Orders | Top row — right |
| Line Chart | order_date (x) · Total Sales (y) | Center left — large |
| Clustered Bar | order_year (x) · Total Sales · Total Profit (y) | Center right |
| Slicer | order_year | Top — filter panel |
| Slicer | Region | Top — filter panel |

#### Step by Step — Line Chart
1. Insert → **Line Chart**
2. X-axis → `order_date` → set to **Month** hierarchy
3. Y-axis → `Total Sales` measure
4. Add `Total Profit` as second Y-axis line
5. Format → Colors → Sales = `#2C3E50`, Profit = `#2ECC71`
6. Format → Title → "Monthly Sales & Profit Trend"

#### KPI Cards Formatting
1. Insert → **Card**
2. Field → drag your measure
3. Format → Callout value → Font size **28pt Bold**
4. Format → Category label → rename to match measure name
5. Format → Background → white, rounded corners

---

### 📄 Page 2 — Regional Analysis

**Purpose:** Identify which regions and countries drive most revenue.

#### Visuals to Add

| Visual Type | Fields | Position |
|-------------|--------|----------|
| Filled Map | country · Total Sales (color) | Center — large |
| Bar Chart | region (x) · Total Sales (y) | Bottom left |
| Bar Chart | region (x) · Profit Margin % (y) | Bottom right |
| Card | Top Region | Top left |
| Card | Regional Sales | Top center |
| Card | Regional Margin % | Top right |
| Slicer | Category | Left panel |
| Slicer | Segment | Left panel |

#### Step by Step — Filled Map
1. Insert → **Map** → choose **Filled Map**
2. Location → `country`
3. Color saturation → `Total Sales` measure
4. Format → Data colors → Minimum = white, Maximum = `#2C3E50`
5. Format → Title → "Sales by Country"

> ⚠️ If map shows wrong countries:
> Column tools → Data category → set `country` to **Country/Region**

---

### 📄 Page 3 — Product Performance

**Purpose:** Identify best and worst performing products and categories.

#### Visuals to Add

| Visual Type | Fields | Position |
|-------------|--------|----------|
| Donut Chart | category · Total Sales | Top left |
| Treemap | sub_category · Total Sales | Top right — large |
| Bar Chart (horizontal) | product_name · Total Profit (top 10) | Center — large |
| Scatter Chart | Total Sales (x) · Profit Margin % (y) · category (legend) | Bottom left |
| Card | Total Quantity | Top KPI |
| Card | Avg Discount % | Top KPI |
| Card | Loss Orders % | Top KPI |
| Slicer | category | Left panel |
| Slicer | order_year | Left panel |

#### Step by Step — Treemap
1. Insert → **Treemap**
2. Category → `sub_category`
3. Values → `Total Sales`
4. Format → Data colors → use categorical palette
5. Format → Title → "Sales by Sub-Category"

#### Step by Step — Top 10 Products Bar
1. Insert → **Bar Chart (horizontal)**
2. Y-axis → `product_name`
3. X-axis → `Total Profit`
4. Filters pane → `product_name` → Filter type = **Top N**
5. Show items = **Top 10** by `Total Profit`

---

### 📄 Page 4 — Customer Segments

**Purpose:** Show RFM segmentation results and top customer analysis.

#### Visuals to Add

| Visual Type | Fields | Position |
|-------------|--------|----------|
| Donut Chart | segment (RFM) · Total Sales | Top left |
| Clustered Bar | segment · Total Customers · Avg Customer Value | Top right |
| Table | customer_name · Total Sales · Total Profit · segment | Center — large |
| Card | Champions Revenue | Top KPI |
| Card | At Risk Revenue | Top KPI |
| Card | Total Customers | Top KPI |
| Slicer | segment (RFM_Segments) | Left panel |
| Slicer | region | Left panel |

#### Step by Step — Customer Table
1. Insert → **Table**
2. Add columns: `customer_name`, `Total Sales`, `Total Profit`,
   `Profit Margin %`, `segment` from RFM_Segments
3. Sort by `Total Sales` descending
4. Format → Conditional formatting → `Total Sales` → Data bars
5. Format → Conditional formatting → `Total Profit` →
   Color scale: red (negative) → green (positive)

---

### 📄 Page 5 — Churn Risk

**Purpose:** Show which customers are at risk and where they are located.

#### Visuals to Add

| Visual Type | Fields | Position |
|-------------|--------|----------|
| Card | Critical Risk Count | Top KPI |
| Card | High Risk Count | Top KPI |
| Card | Revenue at Risk | Top KPI |
| Card | Avg Churn Probability | Top KPI |
| Donut Chart | risk_tier · customer count | Top left |
| Bar Chart | region · Critical Risk Count | Center left |
| Bar Chart | region · Revenue at Risk | Center right |
| Table | customer_name · churn_proba · risk_tier · monetary | Bottom — large |
| Slicer | risk_tier | Left panel |
| Slicer | region | Left panel |

#### Step by Step — Churn Risk Table
1. Insert → **Table**
2. Add: `customer_name`, `churn_proba`, `risk_tier`,
   `recency`, `monetary` from Churn_Scores
3. Sort by `churn_proba` descending
4. Format → Conditional formatting → `churn_proba` →
   Color scale: green (0) → red (1)
5. This gives marketing a prioritised call list

---

### 📄 Page 6 — Sales Forecast

**Purpose:** Show Prophet forecast results imported as a static visual.

#### Option A — Import Chart Image (Simple)
1. Insert → **Image**
2. Browse to `reports/prophet_forecast.png`
3. Resize to fill most of the canvas
4. Insert → **Text Box** → add interpretation notes below

#### Option B — Recreate in Power BI (Advanced)
1. Insert → **Line Chart**
2. Load `prophet_forecast` data by exporting forecast DataFrame to CSV:

```python
# Run this in Notebook 03 to export forecast for Power BI
forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].to_csv(
    '../data/processed/prophet_forecast.csv', index=False
)
```

3. Load `prophet_forecast.csv` into Power BI
4. Line Chart → X = `ds`, Y = `yhat`
5. Add `yhat_lower` and `yhat_upper` as additional lines
6. Format → shade between lines using area chart overlay

> **Recommendation:** Use Option A for speed.
> Use Option B if you want a fully interactive forecast visual.

---

### 🎨 Final Formatting Checklist

```
□ All pages have consistent title font size (16pt Bold)
□ All KPI cards have white background + subtle shadow
□ Color scheme consistent across all pages
□ All slicers are vertical list style (not dropdown)
□ Page navigation buttons added (Insert → Buttons → Navigator)
□ Company/project name in footer on every page
□ Mobile layout configured (View → Mobile Layout)
```

---

### 💾 Publishing Options

| Option | Steps |
|--------|-------|
| **PDF Export** | File → Export → PDF (best for CV/portfolio) |
| **Power BI Service** | Publish → Sign in with free Microsoft account → share link |
| **PNG Screenshots** | File → Export → Export visuals as images |
| **Embed in GitHub** | Export PDF → upload to repo → link in README |

> For your portfolio: Export as PDF and upload to GitHub repo.
> For Kaggle: Take screenshots and embed in notebook markdown.

---
## 🏁 Section 5 — Final Summary & Checklist

### Complete Build Checklist

#### ✅ Data Setup
```
□ superstore_clean.csv loaded into Power BI as Orders table
□ rfm_segments.csv loaded as RFM_Segments table
□ churn_risk_scores.csv loaded as Churn_Scores table
□ Order Date and Ship Date set to Date data type
□ country column data category set to Country/Region
```

#### ✅ Data Model
```
□ RFM_Segments (customer_id) → Orders (customer_id) — One to Many
□ Churn_Scores (customer_id) → Orders (customer_id) — One to Many
□ Both relationships set to Single cross-filter direction
□ No bidirectional relationships
```

#### ✅ DAX Measures Created
```
□ Total Sales
□ Total Profit
□ Total Orders
□ Total Customers
□ Profit Margin %
□ Avg Order Value
□ YoY Sales Growth %
□ Regional Sales
□ Regional Profit
□ Regional Margin %
□ Total Quantity
□ Avg Discount %
□ Avg Profit per Order
□ Loss Orders
□ Loss Orders %
□ Champions Revenue
□ Loyal Revenue
□ At Risk Revenue
□ Lost Revenue
□ Avg Customer Value
□ Critical Risk Count
□ High Risk Count
□ Revenue at Risk
□ Avg Churn Probability
□ Safe Customers
```

#### ✅ Dashboard Pages Built
```
□ Page 1 — Executive Summary   (4 KPI cards + line chart + bar chart)
□ Page 2 — Regional Analysis   (filled map + 2 bar charts + 3 KPI cards)
□ Page 3 — Product Performance (donut + treemap + bar chart + scatter)
□ Page 4 — Customer Segments   (donut + bar chart + customer table)
□ Page 5 — Churn Risk          (4 KPI cards + donut + 2 bar charts + table)
□ Page 6 — Sales Forecast      (prophet chart + interpretation notes)
```

#### ✅ Formatting & Publishing
```
□ Canvas size 1280 × 720 on all pages
□ Consistent color scheme applied
□ All slicers vertical list style
□ Page navigation buttons added
□ Footer with author name on every page
□ Mobile layout configured
□ Exported as PDF → saved to reports/powerbi_dashboard.pdf
□ Screenshots saved to reports/powerbi_page1.png ... page6.png
□ PDF uploaded to GitHub repo
□ Link added to README.md
```

---

### 📁 Files Added After This Notebook

```
reports/
├── powerbi_dashboard.pdf          ← full dashboard export
├── powerbi_page1_executive.png    ← screenshot page 1
├── powerbi_page2_regional.png     ← screenshot page 2
├── powerbi_page3_products.png     ← screenshot page 3
├── powerbi_page4_customers.png    ← screenshot page 4
├── powerbi_page5_churn.png        ← screenshot page 5
└── powerbi_page6_forecast.png     ← screenshot page 6

data/processed/
└── prophet_forecast.csv           ← forecast data for Power BI Page 6
```

---

### 🚀 Git Commit

```bash
git add .
git commit -m "Add Notebook 05 — Power BI Dashboard (6 pages, 25 DAX measures)"
git push origin main
```

---

### ⏭️ What Comes Next

| Notebook | Tool | What We Will Build |
|----------|------|--------------------|
| 06 | SPSS | Statistical analysis — descriptive stats, correlation, regression |
| 07 | R + Shiny | R markdown analysis + interactive Shiny web app |
| 08 | Streamlit | Live Python web app — profit predictor + churn lookup |